# 03 — Calibration de l'estimation de masse**No GPU required.**This is the notebook that fixes the project's weakest claim.`backend/app/mass.py` converts segmented pixel area into grams using a thin-solid heuristic:$$ h_{\text{eff}} = k \cdot \sqrt{A}, \qquad m = A \cdot h_{\text{eff}} \cdot \rho $$The shape factors $k$ currently in that file are **assumed values, not measured ones**. Thisnotebook replaces them with coefficients fitted against weighed reference portions, and reportsthe residual error honestly.Doing this converts "±25–40 %, we think" into "±N %, measured over M samples" — which is thedifference between an assertion and a result.

## 1 — Capture protocolBefore running anything, collect the data. Per class, ten to fifteen portions spanning therealistic range.**Equipment:** a kitchen scale reading to 1 g, a standard ID-1 card (85.60 × 53.98 mm — anycredit or student card), a plain plate, and a phone.**For each portion:**1. Weigh the empty plate. Tare.2. Add the portion. Record the mass in grams.3. Lay the ID-1 card flat on the plate beside the food, fully visible.4. Photograph from directly above, roughly 40 cm, even lighting, no shadow across the card.5. Name the file `<class>_<mass_g>_<n>.jpg` — for example `broccoli_085_03.jpg`.**Why the card matters.** Without a known-size object in frame there is no pixel-to-millimetrescale, and every mass estimate inherits an unknown multiplier. The card is what makes this ameasurement rather than a guess.Upload the folder to `Drive/Nutrivision/calibration/`.

## 2 — Setup

In [ ]:
!pip install -q ultralytics==8.3.55 onnxruntime==1.20.1 \                opencv-python-headless==4.10.0.84 pandas matplotlib seabornimport os, subprocessif os.path.exists("/content/Nutrivision"):    !rm -rf /content/Nutrivisionsubprocess.run(["git","clone","--depth","1",                "https://github.com/Zen-Daitsu/Nutrivision.git",                "/content/Nutrivision"], check=True)%cd /content/Nutrivisionfrom google.colab import drivedrive.mount('/content/drive')ARTIFACTS = "/content/drive/MyDrive/Nutrivision/artifacts"!ls -lh {ARTIFACTS}

In [ ]:
CALIB_DIR = "/content/drive/MyDrive/Nutrivision/calibration"import glob, os, refiles = sorted(glob.glob(f"{CALIB_DIR}/*.jpg") + glob.glob(f"{CALIB_DIR}/*.jpeg"))print(f"{len(files)} calibration photographs")PATTERN = re.compile(r"^([a-z_]+)_(\d+)_(\d+)\.(jpg|jpeg)$", re.I)samples = []for p in files:    m = PATTERN.match(os.path.basename(p))    if m:        samples.append({"path": p, "cls": m.group(1).lower(), "mass_g": float(m.group(2))})    else:        print("skipped, name does not match <class>_<mass>_<n>.jpg :", os.path.basename(p))import pandas as pdcal = pd.DataFrame(samples)print()print(cal.groupby("cls")["mass_g"].agg(["count", "min", "max", "mean"]).round(1))

## 3 — Establish the scale from the fiducialDetect the card and derive pixels per millimetre. The card is a bright quadrilateral of knownaspect ratio (1.586:1), which makes it findable by contour approximation without a detector.

In [ ]:
import cv2, numpy as npCARD_W_MM, CARD_H_MM = 85.60, 53.98CARD_ASPECT = CARD_W_MM / CARD_H_MMdef find_card_px_per_mm(img, tol=0.12):    """Return px_per_mm from the best card-shaped quadrilateral, or None."""    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)    gray = cv2.bilateralFilter(gray, 9, 75, 75)    edges = cv2.Canny(gray, 40, 140)    edges = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)    best, best_err = None, 1e9    for cnt in contours:        if cv2.contourArea(cnt) < 4000:            continue        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)        if len(approx) != 4:            continue        (_, _), (w, h), _ = cv2.minAreaRect(approx)        if min(w, h) < 30:            continue        long_side, short_side = max(w, h), min(w, h)        err = abs(long_side / short_side - CARD_ASPECT) / CARD_ASPECT        if err < tol and err < best_err:            best, best_err = long_side / CARD_W_MM, err    return best# Verify detection on a handful before trusting the batch.import matplotlib.pyplot as pltfig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, row in zip(axes, cal.sample(min(3, len(cal))).itertuples()):    img = cv2.imread(row.path)    ppm = find_card_px_per_mm(img)    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")    ax.set_title(f"{os.path.basename(row.path)}\npx/mm = {ppm:.2f}" if ppm                 else f"{os.path.basename(row.path)}\nCARD NOT FOUND", fontsize=9)plt.tight_layout(); plt.show()

## 4 — Segment each portion and measure its areaUses the trained model. If a portion is missed, the sample is dropped rather than guessed —a calibration fitted on hallucinated masks is worse than no calibration.

In [ ]:
from ultralytics import YOLOimport shutil, osos.makedirs("runs/nutrivision/weights", exist_ok=True)shutil.copy2(f"{ARTIFACTS}/best.pt", "runs/nutrivision/weights/best.pt")model = YOLO("runs/nutrivision/weights/best.pt")from tqdm.auto import tqdmrecords = []for row in tqdm(list(cal.itertuples()), desc="measuring"):    img = cv2.imread(row.path)    ppm = find_card_px_per_mm(img)    if ppm is None:        continue    res = model.predict(row.path, verbose=False, conf=0.30)[0]    if res.masks is None or len(res.masks) == 0:        continue    # Largest mask whose predicted class matches the filename label.    best_area, matched = 0, False    for mask, cls_id in zip(res.masks.data.cpu().numpy(),                            res.boxes.cls.cpu().numpy().astype(int)):        if model.names[cls_id] != row.cls:            continue        area = int(mask.sum())        if area > best_area:            best_area, matched = area, True    if not matched:        continue    # Mask is at model resolution; rescale to original pixel area.    mh, mw = res.masks.data.shape[1:]    oh, ow = img.shape[:2]    area_px_orig = best_area * (oh * ow) / (mh * mw)    px_per_cm = ppm * 10.0    area_cm2 = area_px_orig / (px_per_cm ** 2)    records.append({"cls": row.cls, "mass_g": row.mass_g,                    "area_cm2": round(area_cm2, 2), "px_per_mm": round(ppm, 3)})meas = pd.DataFrame(records)print(f"{len(meas)} of {len(cal)} samples measured successfully")meas.groupby("cls").size()

## 5 — Fit the shape factorWith density $\rho$ fixed from literature, the model reduces to a single free parameter per class:$$ m = A^{1.5} \cdot k \cdot \rho \quad\Longrightarrow\quad k = \frac{m}{A^{1.5} \rho} $$Fitting through the origin by least squares, rather than averaging per-sample $k$, weightslarger portions appropriately — they carry more signal and less relative segmentation noise.

In [ ]:
import numpy as npDENSITY = {   # g/cm3, from USDA and food-science literature    "chicken_breast": 1.04, "beef": 1.05, "egg": 1.03, "white_rice": 0.72,    "quinoa": 0.75, "broccoli": 0.37, "spinach": 0.20, "tomato": 0.95,    "avocado": 0.92, "blueberry": 0.62, "potato": 1.08, "carrot": 0.94,}fits = []for cls, g in meas.groupby("cls"):    rho = DENSITY.get(cls)    if rho is None or len(g) < 4:        print(f"skipping {cls}: {len(g)} samples, density known = {rho is not None}")        continue    x = (g["area_cm2"].values ** 1.5) * rho     # predictor    y = g["mass_g"].values                      # observed mass    k = float((x @ y) / (x @ x))                # least squares through origin    pred = k * x    resid_pct = np.abs(pred - y) / y * 100    fits.append({        "class": cls, "n": len(g), "k": round(k, 4),        "mean_abs_err_pct": round(float(resid_pct.mean()), 1),        "p90_abs_err_pct": round(float(np.percentile(resid_pct, 90)), 1),        "r2": round(float(1 - ((y - pred)**2).sum() / ((y - y.mean())**2).sum()), 3),    })fitted = pd.DataFrame(fits).sort_values("mean_abs_err_pct")fitted

In [ ]:
import matplotlib.pyplot as pltn = len(fitted)if n:    cols = min(3, n); rows = (n + cols - 1) // cols    fig, axes = plt.subplots(rows, cols, figsize=(5.5*cols, 4.5*rows), squeeze=False)    for ax, row in zip(axes.ravel(), fitted.itertuples()):        g = meas[meas["cls"] == row._1]        rho = DENSITY[row._1]        x = (g["area_cm2"].values ** 1.5) * rho        ax.scatter(x, g["mass_g"], color="#7FD1B9", s=45, zorder=3)        xs = np.linspace(0, x.max()*1.05, 50)        ax.plot(xs, row.k*xs, color="#E0715F", lw=2,                label=f"k = {row.k:.3f}\nMAE {row.mean_abs_err_pct}%")        ax.set_xlabel(r"$A^{1.5} \cdot \rho$"); ax.set_ylabel("measured mass (g)")        ax.set_title(row._1); ax.legend(fontsize=8)    for ax in axes.ravel()[n:]:        ax.axis("off")    plt.tight_layout(); plt.show()

## 6 — Emit the calibrated tablePaste the printed dictionary over `SHAPE_FACTOR` in `backend/app/mass.py`, and put the measurederror into the README in place of the estimated range.

In [ ]:
print("# Calibrated against", int(fitted['n'].sum()), "weighed portions")print("# Mean absolute error:", round(float(fitted['mean_abs_err_pct'].mean()), 1), "%")print("SHAPE_FACTOR: dict[str, float] = {")for r in fitted.itertuples():    print(f'    "{r._1}": {r.k:.3f},   # n={r.n}, MAE {r.mean_abs_err_pct}%, R2 {r.r2}')print("}")import json, shutilfitted.to_csv(f"{ARTIFACTS}/mass_calibration.csv", index=False)meas.to_csv(f"{ARTIFACTS}/mass_calibration_raw.csv", index=False)print("\nsaved to", ARTIFACTS)

## What this does and does not establish**Establishes:** the area-to-mass relationship holds for these foods, on a flat plate,photographed from above with a fiducial in frame, over the mass range sampled.**Does not establish:** anything about stacked or layered servings, foods in bowls where depthis hidden, or photographs without a reference card — where the app falls back to afield-of-view assumption and reports `mass_confidence: "low"`.Report the measured error, state the conditions, and let the number stand. An honest ±18 %with a stated protocol is worth more in a defence than an unqualified claim of accuracy.